# Notebook 07 – Mood Overrides

Identify **mood-dependent UI adaptations** supported by evidence.

Mood modifies only UI elements that differ from defaults and are backed by multi-source evidence. It does **not** rebuild the entire interface.

For each mood:
1. Compare UI preferences with the default repository (Notebook 05)
2. Create overrides only when values differ **and** ≥ 2 evidence sources support the change
3. Never include unchanged defaults


## Inputs
- `data/processed/clean_dataset.csv`
- `data/outputs/global_defaults.json`
- `data/outputs/desktop_defaults.json`
- `data/outputs/mobile_defaults.json`
- Notebook 03 statistical results
- Notebook 04 Random Forest + SHAP
- Association rules (mood-only rules, when available)

## Outputs
- `data/outputs/mood_overrides.json`
- `reports/MoodOverrides/mood_overrides.xlsx`


In [1]:
import json
import logging
import sys
from pathlib import Path

import pandas as pd


def _bootstrap_project() -> Path:
    search_from = Path.cwd().resolve()
    candidates = [search_from, *search_from.parents]
    nested_root = search_from / "EvidenceBasedAdaptiveUI"
    if (nested_root / "src" / "config.py").exists():
        candidates.insert(0, nested_root)
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists():
            root = str(candidate)
            if root not in sys.path:
                sys.path.insert(0, root)
            return candidate
    raise FileNotFoundError("Could not find project root containing src/config.py.")


PROJECT_ROOT = _bootstrap_project()

from src.mood_overrides.repository import run_mood_override_pipeline
from src.utils.notebook import setup_notebook

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PATHS, REPORTS = setup_notebook("MoodOverrides")
OUTPUT_DIR = PATHS.data_outputs

print(f"Reports: {REPORTS}")
print(f"JSON output: {OUTPUT_DIR}")


Reports: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/MoodOverrides
JSON output: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs


## Check Inputs


In [2]:
INPUTS = {
    "Clean dataset": PATHS.data_processed / "clean_dataset.csv",
    "Global defaults": OUTPUT_DIR / "global_defaults.json",
    "Desktop defaults": OUTPUT_DIR / "desktop_defaults.json",
    "Mobile defaults": OUTPUT_DIR / "mobile_defaults.json",
    "Statistical results (NB03)": PATHS.reports / "Statistical_Validation" / "tables" / "statistical_results.xlsx",
    "Feature importance (NB04)": PATHS.reports / "Feature_Importance" / "feature_importance.xlsx",
    "SHAP summary (NB04)": PATHS.reports / "Feature_Importance" / "shap_summary.csv",
    "Association rules": PATHS.reports / "AssociationRules" / "base_candidate_rules.csv",
}

for label, path in INPUTS.items():
    print(f"{label}: {'OK' if path.exists() else 'MISSING'}")


Clean dataset: OK
Global defaults: OK
Desktop defaults: OK
Mobile defaults: OK
Statistical results (NB03): OK
Feature importance (NB04): OK
SHAP summary (NB04): OK
Association rules: OK


## Discover Evidence-Backed Overrides

For each mood and UI element:
- Compare mood majority vs default
- Keep only differences
- Require ≥ 2 evidence sources among Statistics / Random Forest / SHAP / Association Rules


In [3]:
result = run_mood_override_pipeline(PROJECT_ROOT, OUTPUT_DIR, REPORTS)

repositories = result.repositories
table = result.table
summary = result.summary

display(table.head(20) if not table.empty else table)


INFO: Mood overrides: 32 total across 8 moods (avg confidence=0.423)


,Mood,UI_Element,Override_Value,Default_Value,Mood_Count,Mood_Share,Confidence,Support,Evidence,Evidence_Count
0,Bored,urgency_pref,"Stock Numbers (Show ""Only 3 left in stock"")",Both (Show both stock and time based urgency),10,0.4545,0.454545,0.044643,"Statistics, Random Forest, SHAP",3
1,Bored,desktop_filter_location,Top Bar (Filters displayed across the top),Sidebar Left (Permanent filter panel on the le...,9,0.4091,0.409091,0.040179,"Random Forest, SHAP",2
2,Bored,desktop_navigation,Sidebar (Permanent navigation panel on the left),Top Bar (Categories always visible across the ...,8,0.3636,0.363636,0.035714,"Statistics, SHAP",2
3,Bored,desktop_price_display,Bold Large (Price is the most prominent element),With Savings Highlighted (Shows discount amoun...,8,0.3636,0.363636,0.035714,"Statistics, Random Forest",2
4,Bored,color_theme_pref,Cool Blues (Blues and grays - calm and trustwo...,"Minimalist Black & White (Clean, high contrast...",6,0.2727,0.272727,0.026786,"Statistics, Random Forest",2
5,Excited,mobile_product_card,"Minimal Clean (Simple image, name, price - no ...","Info Rich (Lots of details, ratings, specs vis...",8,0.8000,0.800000,0.035714,"Statistics, SHAP",2
6,Excited,desktop_image_text_ratio,"Balanced (50% images, 50% text - equal emphasis)","Image-focused (80% images, 20% text - highly v...",7,0.7000,0.700000,0.031250,"Statistics, Random Forest",2
7,Excited,color_theme_pref,"Warm Earthy (Browns, beiges, warm oranges - na...","Minimalist Black & White (Clean, high contrast...",5,0.5000,0.500000,0.022321,"Statistics, Random Forest",2
8,Excited,urgency_pref,"Countdown Timers (Show ""Sale ends in 2 hours"")",Both (Show both stock and time based urgency),5,0.5000,0.500000,0.022321,"Statistics, Random Forest, SHAP",3
9,Happy,desktop_search_visibility,Collapsible (Search bar that expands when clic...,"Always Visible Top (Large, prominent search ba...",14,0.7000,0.764706,0.065000,"Statistics, Random Forest, SHAP, Association R...",4


## Sample Override JSON


In [4]:
sample = next((item for item in repositories if item["n_overrides"] > 0), repositories[0])
print(json.dumps(sample, indent=2, ensure_ascii=False))


{
  "mood": "Bored",
  "n_respondents": 22,
  "n_overrides": 5,
  "overrides": {
    "color_theme_pref": {
      "value": "Cool Blues (Blues and grays - calm and trustworthy)",
      "confidence": 0.272727,
      "support": 0.026786,
      "evidence": [
        "Statistics",
        "Random Forest"
      ]
    },
    "urgency_pref": {
      "value": "Stock Numbers (Show \"Only 3 left in stock\")",
      "confidence": 0.454545,
      "support": 0.044643,
      "evidence": [
        "Statistics",
        "Random Forest",
        "SHAP"
      ]
    },
    "desktop_navigation": {
      "value": "Sidebar (Permanent navigation panel on the left)",
      "confidence": 0.363636,
      "support": 0.035714,
      "evidence": [
        "Statistics",
        "SHAP"
      ]
    },
    "desktop_price_display": {
      "value": "Bold Large (Price is the most prominent element)",
      "confidence": 0.363636,
      "support": 0.035714,
      "evidence": [
        "Statistics",
        "Random Forest"


## Overrides per Mood


In [5]:
for mood, count in summary["overrides_per_mood"].items():
    print(f"{mood}: {count}")


Bored: 5
Excited: 4
Frustrated: 0
Happy: 6
Neutral: 5
Relaxed: 6
Sad: 0
Stressed: 6


## Exports


In [6]:
print("Export locations:")
for name, path in result.export_paths.items():
    print(f"- {name}: {path}")


Export locations:
- mood_overrides_json: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/data/outputs/mood_overrides.json
- mood_overrides_xlsx: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/MoodOverrides/mood_overrides.xlsx
- mood_overrides_csv: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/MoodOverrides/mood_overrides.csv
- summary_md: /Users/mariam/Downloads/New_Master copy/smartshop/EvidenceBasedAdaptiveUI/reports/MoodOverrides/mood_overrides_summary.md


## Final Output


In [7]:
print("Overrides per mood")
for mood, count in summary["overrides_per_mood"].items():
    print(f"  {mood}: {count}")
print("Repository generated.")


Overrides per mood
  Bored: 5
  Excited: 4
  Frustrated: 0
  Happy: 6
  Neutral: 5
  Relaxed: 6
  Sad: 0
  Stressed: 6
Repository generated.
